## Few-Shot & Sequential Prompting - Practice Notebook (Google Colab)
This notebook corresponds to the **06-Few-Shot & Sequential Prompting** documentation page. Follow the concepts and run the code cells.

### 🛠️ Setup Environment & Load API Keys

In [ ]:
# Install core dependencies in Google Colab
!pip install -q langchain-core langchain-google-genai langchain-groq python-dotenv

In [ ]:
import os
import getpass

# Set environment variables dynamically in Colab
if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass.getpass('Enter your Groq API Key: ')
if 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter your Google API Key: ')

### 1. Few-Shot Prompting

Few-Shot prompting provides multiple examples in-context to demonstrate structured output patterns or complex reasoning paths.

#### Example 1: Math Assistant

In [ ]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.chat_models import init_chat_model

# 1. Define list of examples
examples = [
    {"question": "2 + 2", "answer": "4"},
    {"question": "3 + 3", "answer": "6"}
]

# 2. Define example prompt template
example_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="Question: {question}\nAnswer: {answer}"
)

# 3. Create FewShotPromptTemplate
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question: {input}\nAnswer:",
    input_variables=["input"]
)

formatted_prompt = few_shot_prompt.format(input="10 + 10")

# Initialize model and invoke
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
response = llm.invoke(formatted_prompt)
print(response.content)

#### Example 2: Sentiment Classifier

In [ ]:
# 1. Define few-shot examples
examples_sent = [
    {"text": "I love this product.", "sentiment": "Positive"},
    {"text": "This is terrible.", "sentiment": "Negative"},
    {"text": "This movie is amazing.", "sentiment": "Positive"}
]

# 2. Define example prompt template
example_prompt_sent = PromptTemplate(
    input_variables=["text", "sentiment"],
    template="Text: {text}\nSentiment: {sentiment}"
)

# 3. Create FewShotPromptTemplate
few_shot_prompt_sent = FewShotPromptTemplate(
    examples=examples_sent,
    example_prompt=example_prompt_sent,
    suffix="Text: {input}\nSentiment:",
    input_variables=["input"]
)

formatted_prompt_sent = few_shot_prompt_sent.format(input="The customer support was extremely unhelpful and rude.")
response = llm.invoke(formatted_prompt_sent)
print(response.content.strip())

#### Exercise: Few-Shot Entity Extraction
Create a few-shot prompt using `FewShotPromptTemplate` that formats examples for extracting a person and their company from text.

<details>
<summary><b>Click to view solution</b></summary>

```python
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

examples = [
    {"text": "John works at Google.", "output": '{"person": "John", "company": "Google"}'},
    {"text": "Alice joined Microsoft.", "output": '{"person": "Alice", "company": "Microsoft"}'}
]

example_prompt = PromptTemplate(
    input_variables=["text", "output"],
    template="Text: {text}\nOutput: {output}"
)

few_shot_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Text: {input}\nOutput:",
    input_variables=["input"]
)

formatted_prompt = few_shot_template.format(input="Bob works at Amazon.")
response = llm.invoke(formatted_prompt)
print(response.content)
```
</details>

In [ ]:
# Write your code here

### 2. Sequential Prompting

Sequential prompting chains multiple prompts together so the output of one step feeds directly as an input variable into the next.

#### Sequential Chains using LCEL

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# First Prompt: Generates detailed information
info_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Provide a detailed overview of {topic}."
)

# Second Prompt: Summarizes information generated by Prompt 1
summary_prompt = PromptTemplate(
    input_variables=["details"],
    template="Summarize the following details in 2 bullet points:\n{details}"
)

# Compose sequential chain
sequential_chain = (
    {"details": info_prompt | llm | StrOutputParser()}
    | summary_prompt
    | llm
    | StrOutputParser()
)

response = sequential_chain.invoke({"topic": "Docker"})
print(response)

#### Exercise: Sequential Learning Planner
Write a sequential chain that takes a goal activity (e.g., `learn to swim`), asks the LLM to write a comprehensive learning guide, and then passes that guide to a second prompt that formats it as a 1-week crash course schedule.

<details>
<summary><b>Click to view solution</b></summary>

```python
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

learning_prompt = PromptTemplate(
    input_variables=["activity"],
    template="I want to learn how to {activity}. Can you suggest how I can learn this step-by-step?"
)

time_prompt = PromptTemplate(
    input_variables=["learning_plan"],
    template="I only have one week. Can you create a concise plan to help me hit this goal: {learning_plan}."
)

seq_chain = (
    {"learning_plan": learning_prompt | llm | StrOutputParser()}
    | time_prompt
    | llm
    | StrOutputParser()
)

result = seq_chain.invoke({"activity": "learn to swim"})
print(result)
```
</details>

In [ ]:
# Write your code here